# ShiftRoute-RL v0.3 — Verified cost versus safety

This notebook independently reads recorded **synthetic** decisions, compares paired seed-level outcomes, and visualizes the security-versus-cost frontier. Results are **not** from a pretrained LLM and do **not** establish new general guarantees. See `docs/research-v03.md`.

Primary claims: pure score routing saves review cost but violates the strict zero-increase constraint under adversarial shift; an external sound capability proof can safely bypass some costly reviews under the same reviewer outcomes; proof unsoundness removes that guarantee.

In [ ]:
from pathlib import Path
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
root = Path.cwd()
if not (root / "results/v03/experiment.json").exists():
    root = root.parent
assert (root / "results/v03/experiment.json").exists(), "Run the v0.3 experiment first"
raw=json.loads((root / "results/v03/experiment.json").read_text())
records=pd.DataFrame(raw["records"])
summary=pd.DataFrame(raw["summary"])
print(f"{len(records)} recorded comparisons; {records.stream_seed.nunique()} seeds; {records.regime.nunique()} regimes")

In [ ]:
hard=summary[(summary.regime=="joint-shift") & (summary.sensitivity==1.0)].copy()
cols=["mode","unsafe_executed_fraction","review_fraction","review_cost_per_action","legitimate_executed_fraction"]
print(hard[cols].to_string(index=False, float_format=lambda x:f"{x:.4f}"))
base=float(hard.loc[hard["mode"]=="all","review_cost_per_action"].iloc[0])
proof=float(hard.loc[hard["mode"]=="proof","review_cost_per_action"].iloc[0])
print("Proof-first synthetic relative cost reduction:", round((base-proof)/base*100,2), "%")
assert abs(float(hard.loc[hard["mode"]=="proof","unsafe_executed_fraction"].iloc[0])) < 1e-12

In [ ]:
fig, ax=plt.subplots(figsize=(8,5))
for mode in ("all","threshold","sentinel","adaptive","proof"):
    row=hard[hard["mode"]==mode].iloc[0]
    x,y=row.review_cost_per_action,100*row.unsafe_executed_fraction
    ax.scatter([x],[y],s=75)
    ax.annotate(mode,(x,y),xytext=(5,7),textcoords="offset points")
ax.set(xlabel="Mean synthetic review cost / action",ylabel="Unsafe executions / proposals (%)",title="Five-seed means: held-out joint shift, perfect expensive reviewer")
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

In [ ]:
stress=pd.DataFrame(json.loads((root / "results/v03/proof_stress.json").read_text())["summary"])
print(stress[["reviewer_sensitivity","certificate_false_accept_probability","unsafe_executed_rate_mean","review_cost_per_action_mean"]].to_string(index=False,float_format=lambda x:f"{x:.5f}"))

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for sens in [1.0,.94]:
    d=stress[stress.reviewer_sensitivity==sens]
    ax.plot(100*d.certificate_false_accept_probability,100*d.unsafe_executed_rate_mean,marker="o",label=f"Reviewer sensitivity {sens:.0%}")
ax.set(xlabel="Incorrect certificate issuance rate on unsafe proposals (%)",ylabel="Unsafe executions / all proposals (%)",title="A proof only helps safely if its soundness assumption holds")
ax.legend();ax.grid(alpha=.2);plt.tight_layout();plt.show()

## Interpretation and external validity

The safe proof-first finding is a **conditional, pointwise identity**: if all unsafe proposals lack trusted certificates and the same reviewer makes the same decisions when consulted, skipping review for certified-safe actions cannot add unsafe executions. This requires the independent certificate issuer to be correct. The simulation grants it access to latent authorization truth, which would not exist for a generic RL agent.

When an auditor is fallible, complete verification already fails sometimes, and proof-first **matches rather than improves** that baseline. In these experiments, adaptive sentinel auditing is cheaper than verify-all but does increase unsafe executions, so it **fails the user's strict safety constraint**.

This notebook should not be used to claim an LLM-scale result, an established novel algorithm, production savings, or peer-reviewed discovery.